# Modelado — tfm_energia

Punto de partida para entrenar y comparar modelos. Este notebook NO reconstruye el dataset --
llama a `construir_dataset_maestro.py` (ver `docs/` o el manual del script), que ya deja el
target correctamente alineado a D+1, las features sin fuga, y el split oficial del equipo.

Rama: `willy_test` · Fuente: `modelos/construir_dataset_maestro.py`

## 0. Cargar el dataset maestro y dividirlo

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "modelos"))

import pandas as pd
from construir_dataset_maestro import construir_dataset_diario, dividir_train_val_test

dataset = construir_dataset_diario()

# Separar target de features -- esto no lo hace el script, hay que hacerlo aqui
TARGET_COLS = [c for c in dataset.columns if c.startswith("price_h")]
FEATURE_COLS = [c for c in dataset.columns if c not in TARGET_COLS]

train, val, test = dividir_train_val_test(dataset)

X_train, y_train = train[FEATURE_COLS], train[TARGET_COLS]
X_val,   y_val   = val[FEATURE_COLS],   val[TARGET_COLS]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET_COLS]

print(f"dataset: {dataset.shape[0]} dias x {dataset.shape[1]} columnas ({len(FEATURE_COLS)} features + {len(TARGET_COLS)} target)")
print(f"train: {X_train.shape[0]} dias | validation: {X_val.shape[0]} dias | test: {X_test.shape[0]} dias")

D:\POSGRADO\TFM\edev_models\ingesta\construir_dataset_maestro.py:437: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  target_wide = pd.read_sql(


D:\POSGRADO\TFM\edev_models\ingesta\construir_dataset_maestro.py:173: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


D:\POSGRADO\TFM\edev_models\ingesta\construir_dataset_maestro.py:196: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


D:\POSGRADO\TFM\edev_models\ingesta\construir_dataset_maestro.py:225: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


D:\POSGRADO\TFM\edev_models\ingesta\construir_dataset_maestro.py:241: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


...

D:\POSGRADO\TFM\edev_models\ingesta\construir_dataset_maestro.py:290: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_comm = pd.read_sql(
D:\POSGRADO\TFM\edev_models\ingesta\construir_dataset_maestro.py:297: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_capd = pd.read_sql(
D:\POSGRADO\TFM\edev_models\ingesta\construir_dataset_maestro.py:320: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_load = pd.read_sql(


D:\POSGRADO\TFM\edev_models\ingesta\construir_dataset_maestro.py:325: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_eolica = pd.read_sql(


D:\POSGRADO\TFM\edev_models\ingesta\construir_dataset_maestro.py:330: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_solar = pd.read_sql(


D:\POSGRADO\TFM\edev_models\ingesta\construir_dataset_maestro.py:379: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


dataset: 2410 dias x 121 columnas (97 features + 24 target)
train: 1822 dias | validation: 364 dias | test: 224 dias


## 1. Próximos pasos

- [ ] Imputar nulos (mediana de `X_train`, nunca de val/test)
- [ ] Baseline: persistencia (D+1 = D-1) y/o ARIMA sobre `price_h*`
- [ ] Primer GBM (Random Forest / XGBoost) sobre train, evaluado en validation
- [ ] Comparar arquitecturas en validation; tocar test una sola vez, al final

## 2. Eventos extremos — tratamiento distinto segun el tipo (ver `docs/notas_memoria_tfm.md` nota 6
y el manual de `construir_dataset_maestro.py`, seccion 07)

El historico incluye dos anomalias reales, y **no se tratan igual**:

- **Apagon 28-abril-2025**: caso unico, sin repeticion -- no generaliza, solo se memorizaria. Se
  aparta del entrenamiento y de la evaluacion (esa fila y las que arrastran su lag via D-1/D-7).
- **Crisis energetica 2021-2022**: cambio sostenido, con cientos de dias de ejemplo -- si se puede
  aprender de el. No se aparta -- se modela a traves de su causa (precio del gas, ya incluido como
  feature continua en el dataset), nunca con una etiqueta "esto fue crisis" que no generalizaria a
  la proxima subida del gas por otro motivo.

Pendiente: el test actual (2026) no cubre ninguna crisis parecida -- falta una prueba de estres
aparte sobre 2021-2022 antes de dar cualquier modelo por bueno.